## Code For Zeroshot

In [ ]:
%%time

import os
import json
import sys
from pathlib import Path
from typing import Dict, Any, List, Optional, Set, Tuple

import torch
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

from transformers import AutoModelForCausalLM
from deepseek_vl2.models import DeepseekVLV2Processor, DeepseekVLV2ForCausalLM
from deepseek_vl2.utils.io import load_pil_images

# ---------------- Config ----------------
INPUT_PATH   = "../MedGemma/test_flat.jsonl"      # your test JSONL
OUTPUT_CSV   = "deepseek_vl2_predictions.csv"     # output file

# Variants: "deepseek-ai/deepseek-vl2-tiny", "-small", or local folder "./models/deepseek-vl2-tiny"
MODEL_PATH   = "./models/deepseek-vl2-tiny"

DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE        = torch.float32   # keep FP32 for now to avoid dtype mismatch

MAX_WORKERS       = 1          # keep 1 while debugging; you can increase later
BATCH_SAVE_EVERY  = 25
DEBUG_SANITY      = True       # run a one-sample sanity check before full run

print("Using device:", DEVICE)
print("Using dtype:", DTYPE)

# --------------- Load model ----------------
print("Loading DeepSeek-VL2 model...")
vl_chat_processor: DeepseekVLV2Processor = DeepseekVLV2Processor.from_pretrained(MODEL_PATH)
tokenizer = vl_chat_processor.tokenizer

vl_gpt: DeepseekVLV2ForCausalLM = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=DTYPE,   # load weights in FP32
)
vl_gpt = vl_gpt.to(DEVICE).eval()
print("Model loaded.")


# --------------- Helpers ----------------
def load_jsonl(path: str | Path) -> List[Dict[str, Any]]:
    recs: List[Dict[str, Any]] = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            recs.append(json.loads(line))
    return recs


def autocorrect_path(p: Path) -> Path:
    """Fix common Unannoated/Unannotated typos in your RadSpineXR paths."""
    if p.exists():
        return p
    s = str(p)
    fixes = {
        "Unannoated": "Unannotated",
        "unannoated": "unannotated",
        "Unannoated_images": "Unannotated_images",
    }
    for bad, good in fixes.items():
        if bad in s:
            q = Path(s.replace(bad, good))
            if q.exists():
                return q
    return p


@torch.inference_mode()
def ask_deepseek_vl2(image_path: str, question: str) -> str:
    """
    True multimodal call: open image locally and pass it with text question
    using the official DeepSeek-VL2 processor + model.

    Follows the README pattern:
      - <|User|> role
      - content starts with "<image>\\n" then your question
      - "images": [image_path]
    """

    # ----- 1) Build conversation in the official format -----
    conversation = [
        {
            "role": "<|User|>",
            "content": f"<image>\n{question}",
            "images": [image_path],
        },
        {
            "role": "<|Assistant|>",
            "content": "",
        },
    ]

    # ----- 2) Load image(s) -----
    pil_images = load_pil_images(conversation)
    if not pil_images or pil_images[0] is None:
        raise RuntimeError(f"Failed to load image via load_pil_images: {image_path}")

    # ----- 3) Preprocess inputs -----
    prepare_inputs = vl_chat_processor(
        conversations=conversation,
        images=pil_images,
        force_batchify=True,
        system_prompt="You are a careful medical radiology assistant. "
                      "Answer concisely and clinically about spinal X-ray findings.",
    ).to(vl_gpt.device, dtype=DTYPE)

    # ----- 4) Encode vision → embeddings -----
    inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)

    # ----- 5) Generate answer using the multimodal generate -----
    outputs = vl_gpt.generate(
        inputs_embeds=inputs_embeds,
        input_ids=prepare_inputs.input_ids,
        images=prepare_inputs.images,
        images_seq_mask=prepare_inputs.images_seq_mask,
        images_spatial_crop=prepare_inputs.images_spatial_crop,
        attention_mask=prepare_inputs.attention_mask,
        pad_token_id=tokenizer.eos_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=256,
        do_sample=False,
        use_cache=True,
    )

    # ----- 6) Extract only the new tokens (assistant answer) -----
    full_ids = outputs[0].cpu().tolist()
    prompt_len = len(prepare_inputs.input_ids[0])
    gen_ids = full_ids[prompt_len:]
    answer = tokenizer.decode(gen_ids, skip_special_tokens=True)

    return answer.strip()


def process_example(ex: Dict[str, Any], done: Set[Tuple[str, str]]) -> Optional[Dict[str, Any]]:
    img = ex.get("image_path") or ex.get("image") or ex.get("img_path")
    q   = ex.get("question")   or ex.get("prompt") or ex.get("query")
    gt  = ex.get("answer")     or ex.get("ground_truth") or ""

    if not img or not q:
        sys.stderr.write(f"[WARN] Malformed example: keys={list(ex.keys())}\n")
        return None

    p = autocorrect_path(Path(img))
    row_key = (str(p), str(q))

    # Resume-safe: skip if already in CSV
    if row_key in done:
        return None

    if not p.exists():
        pred = f"[ERROR] Image not found: {p}"
    else:
        try:
            pred = ask_deepseek_vl2(str(p), q)
        except Exception as e:
            pred = f"[ERROR] Inference failed: {e}"

    return {
        "image_path": str(p),
        "question": q,
        "answer": gt,
        "generated_answer": pred,
    }


# --------------- Sanity check (one sample) ----------------
def sanity_check(records: List[Dict[str, Any]]):
    if not records:
        print("[SANITY] No records found in JSONL.")
        return

    ex = records[0]
    img = ex.get("image_path") or ex.get("image") or ex.get("img_path")
    q   = ex.get("question")   or ex.get("prompt") or ex.get("query")
    gt  = ex.get("answer")     or ex.get("ground_truth") or ""

    if not img or not q:
        print("[SANITY] First example malformed, skipping.")
        return

    p = autocorrect_path(Path(img))
    print("[SANITY] First example:")
    print("  Raw image_path:", img)
    print("  Resolved path :", p)
    print("  Exists?       :", p.exists())
    print("  Question      :", q)

    if not p.exists():
        print("[SANITY] Image file not found → fix paths before full run.")
        return

    # try actual model call
    try:
        ans = ask_deepseek_vl2(str(p), q)
        print("  Sample answer:", ans[:200], "..." if len(ans) > 200 else "")
    except Exception as e:
        print("[SANITY] Inference error:", e)


# --------------- Main runner ----------------
def run():
    records = load_jsonl(INPUT_PATH)
    print(f"Loaded {len(records)} examples from {INPUT_PATH}")

    if DEBUG_SANITY:
        sanity_check(records)
        print("[SANITY] Done sanity check; continuing to full dataset.\n")

    rows: List[Dict[str, Any]] = []
    out_path = Path(OUTPUT_CSV)

    # Resume-safe
    done: Set[Tuple[str, str]] = set()
    if out_path.exists():
        prev = pd.read_csv(out_path)
        for _, r in prev.iterrows():
            done.add((str(r["image_path"]), str(r["question"])))
        rows = prev.to_dict("records")
        print(f"Resuming from {OUTPUT_CSV}: {len(done)} rows already present")

    futures = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for ex in records:
            futures.append(executor.submit(process_example, ex, done))

        for fut in tqdm(as_completed(futures), total=len(futures), desc="Inferencing (DeepSeek-VL2)"):
            row = fut.result()
            if row is None:
                continue

            rows.append(row)

            # periodic save
            if len(rows) % BATCH_SAVE_EVERY == 0:
                pd.DataFrame(
                    rows,
                    columns=["image_path", "question", "answer", "generated_answer"],
                ).to_csv(out_path, index=False)

    # final save
    pd.DataFrame(
        rows,
        columns=["image_path", "question", "answer", "generated_answer"],
    ).to_csv(out_path, index=False)

    print(f"✅ Done. Wrote {len(rows)} rows → {out_path.resolve()}")


run()
